# Delivery Regression · Data Preparation Audit

## tl;dr
The course archive and 96,470-order completed-delivery cohort are independently verified.
The 16-input schema contains no outcome values, but quote/catalogue/order snapshots require explicit
purchase-time assumptions. Legacy postal means are contaminated by 29 clearly implausible geographic
rows. A screened, deduplicated postal-median input preview has been checked against the repaired formal preparation in `versions/preparation_v2/`,
without changing the cohort or fitting models. Old MAE/OOF artifacts still refer to legacy geography and require a later refit
before handoff. Event chronology contradictions in 84 orders are flagged rather than automatically
changing otherwise non-negative purchase-to-receipt targets.


## Context & Methods
Audit the preparation underlying the modelling Notebook cells 1–5, starting from the seven raw course
CSVs rather than assuming agreement with Phase 1 proves correctness. Grain: one order. Target:
fractional purchase-to-receipt days conditional on completed deliveries. This is a data audit, not
parameter tuning; models and final-test errors are not used to choose cleaning rules.

### Key Assumptions
The archived estimate represents the original purchase quote; basket, seller selection, destination
and charges reflect the placed order; catalogue and postal locations are static historical proxies.
No revision timestamps exist to prove these assumptions. Final status is a cohort rule, never an input.
The geographic screening envelope is a generous engineering rule informed by IBGE national extremes
(including Atlantic islands), not a country polygon. No external coordinates or newer Olist data are
used. Postal medians remain proxies and cannot correct every mislabelled address.


## Data
### Load the unchanged course archive and run independent checks
Audits save summary tables under `outputs/preparation_audit/`. Order-level flags and candidate inputs
remain under ignored `data/preparation_audit/`. The source script does not overwrite existing model
inputs, predictions, OOF, or pipelines. This companion uses the same local environment as the model Notebook.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from data_preparation_audit import run_audit
result = run_audit()
display(result['sources'][['table','rows','columns']])


,table,rows,columns
0,orders,99441,8
1,items,112650,7
2,customers,99441,5
3,sellers,3095,4
4,products,32951,9
5,geo,1000163,5
6,translation,71,2


## Results
### 1. Order eligibility and joins
Primary/composite keys and foreign keys pass. Rebuild eligibility independently from all orders:
delivered status, observed purchase/receipt, non-negative duration, available item records.
All delivered orders have item records, so this requirement does not lose completed targets.
No missing target is imputed, and all 306 valid >60-day durations remain.


In [2]:
display(result['cohort_flow'])
display(pd.read_csv(ROOT / 'outputs/preparation_audit/integrity_checks.csv'))


,stage,n
0,all source orders,99441
1,delivered status,96478
2,delivered with purchase/receipt,96470
3,non-negative observed target,96470
4,also has item records,96470


,check,affected_rows
0,orders primary key,0
1,items primary key,0
2,customers primary key,0
3,sellers primary key,0
4,products primary key,0
5,translation primary key,0
6,order_id FK coverage,0
7,product_id FK coverage,0
8,seller_id FK coverage,0
9,customer_id FK coverage,0


### 2. Event chronology and missingness
Receipt never precedes purchase. However, 61 eligible receipts precede approval and 23 precede carrier
handoff (84 distinct orders). These other event timestamps are not inputs or components of the target;
flag for sensitivity review rather than assert purchase/receipt is wrong. Eight delivered receipts are
missing and excluded. Small geographic/weight missingness is preserved with indicators and fold-only
imputation, rather than silently dropping otherwise eligible orders.


In [3]:
display(result['chronology'])
display(result['feature_profile'][['field','missing_n','missing_pct','unique_n']].round(3))


,check,n
0,receipt before purchase,0
1,eligible receipt before approval,61
2,eligible receipt before carrier,23
3,eligible union event contradictions,84
4,non-delivered status with receipt,6


,field,missing_n,missing_pct,unique_n
0,promised_lead_time_days,0,0.000,93880
1,max_distance_km,476,0.493,19169
2,total_price,0,0.000,7522
3,total_freight,0,0.000,7716
4,freight_ratio,0,0.000,43366
5,item_count,0,0.000,17
6,seller_count,0,0.000,5
7,avg_product_weight_g,22,0.023,2766
8,any_distance_missing,0,0.000,2
9,any_weight_missing,0,0.000,2


### 3. Exact input calculations and availability
All non-geographic inputs match independent full-cohort reconstruction: item totals/counts,
all-seller state relation, mixed/missing category logic, positive weights, quote and calendar.
Item price is positive, freight non-negative. High freight/price ratios are not automatically invalid.
Missing/untranslated categories are explicit groups, not silent order exclusions. There is no column
value derived from actual duration in X. Availability of archived commercial snapshots is conditional,
not proven merely because a join succeeds. Future implementation needs the original quote at the
chosen prediction instant, or must remove it/change the prediction instant explicitly.


In [4]:
display(result['availability'][['field','availability_evidence']])
display(result['feature_profile'].drop(columns=['n','unique_n']).round(3))


,field,availability_evidence
0,promised_lead_time_days,conditional archived quote
1,max_distance_km,static reference assumption
2,total_price,conditional order snapshot
3,total_freight,conditional order snapshot
4,freight_ratio,conditional order snapshot
5,item_count,conditional order snapshot
6,seller_count,conditional order snapshot
7,avg_product_weight_g,static catalogue assumption
8,any_distance_missing,static reference assumption
9,any_weight_missing,static catalogue assumption


,field,missing_n,missing_pct,min,median,p99,max
0,promised_lead_time_days,0,0.000,2.008,23.231,50.211,155.135
1,max_distance_km,476,0.493,0.000,435.600,2483.900,8677.900
2,total_price,0,0.000,0.850,86.500,990.000,13440.000
3,total_freight,0,0.000,0.000,17.170,104.246,1794.960
4,freight_ratio,0,0.000,0.000,0.224,1.459,21.447
5,item_count,0,0.000,1.000,1.000,3.000,21.000
6,seller_count,0,0.000,1.000,1.000,2.000,5.000
7,avg_product_weight_g,22,0.023,2.000,700.000,18229.240,40425.000
8,any_distance_missing,0,0.000,0.000,0.000,0.000,1.000
9,any_weight_missing,0,0.000,0.000,0.000,0.000,1.000


### 4. Geographic repair preview
Global latitude/longitude domains alone miss the errors: 29 rows fall outside a generous Brazil envelope
(lat −35 to 6, lon −75 to −25). Legacy means create four eligible distances above 4,500 km, reaching
8,677.9 km. The 4,500 threshold is a diagnostic, not an order exclusion rule.

Compare legacy mean, screened mean, screened unique-coordinate median, and a stricter postal+state
median. The recommended candidate is screened postal median: max 3,398.5 km, 477 missing distances
versus 476 previously, and 154 orders changed by >=50 km. Median absolute change is 0.1 km.
Strict state lookup creates 560 newly missing order distances, so it is not silently adopted; eight
multi-state postal prefixes and uncertain label consistency need separate caution.
No model scores were used to choose the geographic candidate. The candidate changes only distance
and its missingness flag, not targets or cohorts. Source duplicate reference points are not extra
orders and need not receive repeated weight in a representative-coordinate proxy.


In [5]:
display(pd.Series(result['summary']['geography']).to_frame('value'))
display(result['geography'].round(3))


,value
source_rows,1000163
exact_duplicate_rows,261831
postal_prefixes,19015
multi_state_prefixes,8
outside_global_domain,0
outside_conservative_brazil_envelope,29
flagged_postal_prefixes,20


,policy,eligible_n,missing_max_distance_n,any_distance_missing_n,max_observed_distance_km,above_4500_km_n,changed_ge_50_km_n,median_absolute_change_km,new_missing_n
0,legacy_mean,96470,476,480,8677.9,4,0,0.0,0
1,screened_mean,96470,477,481,3399.2,0,26,0.0,1
2,screened_median,96470,477,481,3398.5,0,154,0.1,1
3,screened_state_median,96470,1036,1056,3398.5,0,159,0.1,560


### 5. Existing partitions and next-stage readiness
Recomputed chronological assignments match the saved manifest; each development training receipt is
known before its cutoff. This audit does not change dates to improve a score. Training CV remains
conditional on labels available by March 1, a limitation for slow recent orders; its suitability for
next-stage selection should be assessed separately. No final-test predictions or metrics are generated.


In [6]:
display(pd.Series({
    'cohort_verified':result['summary']['target_matched_all_orders'],
    'non_geographic_inputs_verified':result['summary']['non_geographic_inputs_matched_all_orders'],
    'split_manifest_verified':result['summary']['existing_splits_verified'],
    'candidate_applied_to_training':False,
    'models_fitted_this_audit':False,
    'test_scored_this_audit':False,
}).to_frame('status'))


,status
cohort_verified,True
non_geographic_inputs_verified,True
split_manifest_verified,True
candidate_applied_to_training,False
models_fitted_this_audit,False
test_scored_this_audit,False


### 6. Materialize the corrected preparation without model fitting
The formal `prepare_data` default now uses screened postal medians. Write the full prepared v2
cohort to `versions/preparation_v2/`, leaving legacy inputs/fits/predictions untouched. Independently
check the candidate's distance and all 16 inputs against this actual implementation. Targets and split
assignments remain identical. A later model run must regenerate comparisons and OOF together.


In [7]:
from delivery_regression import prepare_data, FEATURES
v2, v2_splits, v2_quality = prepare_data(output_root=ROOT / 'versions/preparation_v2')
preview = pd.read_csv(ROOT / 'data/preparation_audit/screened_median_input_preview.csv').set_index('order_id')
formal = v2.set_index('order_id')[FEATURES].reindex(preview.index)
pd.testing.assert_frame_equal(formal, preview[FEATURES], check_dtype=False, rtol=1e-9, atol=1e-9)
legacy = pd.read_csv(ROOT / 'data/eligible_orders.csv').set_index('order_id')
pd.testing.assert_series_equal(v2.set_index('order_id').lead_time_days.reindex(legacy.index), legacy.lead_time_days)
pd.testing.assert_series_equal(v2.set_index('order_id')['split'].reindex(legacy.index), legacy['split'])
display(pd.Series({k:v2_quality[k] for k in ['preparation_version','geography_policy','eligible_orders',
                                          'retained_over_60_days','missing_max_distance']}).to_frame('value'))

import json
result['summary'].update({'production_preparation_corrected':True,
                         'formal_v2_materialized':'versions/preparation_v2/data/eligible_orders.csv',
                         'status':'cohort and corrected geographic preparation verified; legacy model artifacts await refit'})
_ = (ROOT / 'outputs/preparation_audit/audit_summary.json').write_text(json.dumps(result['summary'],indent=2)+'\n')


,value
preparation_version,purchase_inputs_v2_screened_median
geography_policy,screened_median
eligible_orders,96470
retained_over_60_days,306
missing_max_distance,477


## Takeaways
Keep the target, current completed-delivery eligibility, 16-field initial allowlist and date manifests.
Do not declare cells 1–5 final without repairing geography and clarifying availability assumptions.
Retain/flag the 84 event contradictions; do not delete valid slow orders to improve MAE.
The reviewed geographic recipe is now integrated into formal `prepare_data` and materialized in
`versions/preparation_v2/data/eligible_orders.csv`; the independent preview matches its 16 inputs. Regenerate data-version metadata, comparison tables,
OOF and fitted pipelines together after integration. Existing legacy-model handoff is provisional.

## Sources and AI use
Course `IT5006_Project-Data.zip`; exact seven member hashes are in the saved source inventory.
IBGE Anuário Estatístico do Brasil 2022 table 1.1.1.1 and island notes:
https://biblioteca.ibge.gov.br/visualizacao/periodicos/20/aeb_2022.pdf (geographic validation rationale only).
AI assisted audit implementation and interpretation; checks execute against source records.
